# Phase 2 · Step 1 — Qdrant local mode + collection

Gets Qdrant running in local (embedded) mode and creates a collection
sized for `BAAI/bge-small-en-v1.5` (384 dimensions), the embedding model
Step 2 will use. See
`learning/04_rag_pipeline/01_qdrant_setup.md` for the why behind these
choices.

In [1]:
from pathlib import Path

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams

EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"
EMBEDDING_DIM = 384
COLLECTION_NAME = "papermind_chunks"
STORAGE_PATH = Path("data/qdrant")

client = QdrantClient(path=str(STORAGE_PATH))
client

## Create the collection

`VectorParams` fixes the collection's vector size and distance metric up
front — every point added later must match this shape.

In [2]:
if not client.collection_exists(COLLECTION_NAME):
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(size=EMBEDDING_DIM, distance=Distance.COSINE),
    )

print(f"Collection exists: {client.collection_exists(COLLECTION_NAME)}")

Collection exists: True


## Verify

Confirm the collection was created with the right vector size and distance
metric, and that it starts out empty.

In [3]:
info = client.get_collection(COLLECTION_NAME)
print("Vector size:", info.config.params.vectors.size)
print("Distance:", info.config.params.vectors.distance)
print("Points count:", info.points_count)

Vector size: 384
Distance: Cosine
Points count: 0


## Next

Phase 2 · Step 2 embeds the chunks from Phase 1 · Step 2 with `fastembed`
and upserts them into this collection, each point carrying the chunk's
`page_label` in its payload so search results can still cite a page.